# CNN MVP – altura de edificações (8 cidades brasileiras)
Regressão do rótulo escalar `height_m` a partir de patches Sentinel-1/2 (48 canais). Todos os modelos aprendem com a loss RMSE.
Fluxo: setup → dados → folds → augmentation/balanceamento → pré-processamento → treino base → tuning → arquiteturas em U → hold-out → **teste (final)**.

**v8:** folds do `folds.csv` (os mesmos dos modelos clássicos), validação interna separada do hold-out, banco e pastas próprios (`runs_v8/`).
**Retomada:** se a execução cair, rode o notebook de novo: o run incompleto é reaberto e só as iterações que faltam são treinadas.

## 1. Setup e GPU

In [ ]:
# Setup: paths and shared folds (cnn/cnn_setup.py). Run this notebook from the repository root or from cnn/notebooks.
import sys, pathlib
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent, pathlib.Path.cwd() / "cnn"):
    if (_p / "cnn_setup.py").exists():
        sys.path.insert(0, str(_p)); break
import cnn_setup as cs

In [ ]:
# pip install "tensorflow[and-cuda]" pandas numpy mlflow   (instala CUDA e cuDNN compatíveis)
import gc, glob, hashlib, json, os, shutil, zlib
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf, mlflow
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

info = tf.sysconfig.get_build_info()
print("TF", tf.__version__, "| CUDA", info.get("cuda_version"), "| cuDNN", info.get("cudnn_version"))
print("GPUs:", gpus)

## 2. Configuração e MLflow

In [ ]:
TRAIN_DIR = str(cs.TRAIN_DIR)
TEST_DIR = str(cs.TEST_DIR)
TRAIN_PATTERN = "*.tfrecord.gz"                                       # todos os shards da pasta
TEST_PATTERN = "*.tfrecord.gz"
DIST_GLOB = str(cs.DIST_GLOB)                   # distâncias teste→treino, para o teste "clean"

C_IN, S, NODATA = 48, 33, -9999.0
PATCH_PX = 11                                                         # ímpar, mínimo 5 para esta arquitetura
K_FOLDS, OUTER_FOLD = 5, 0                                            # OUTER_FOLD: hold-out, só para relatar
INNER_VAL_FOLD = (OUTER_FOLD + 1) % K_FOLDS                           # validação interna: early stopping, tuning, seleção
BATCH, EPOCHS, LR = 128, 100, 1e-3
PRED_BATCH = 128                                                      # batch de inferência; reduza se faltar memória na GPU
W_REF, W_MAX = 10.0, 5.0                                              # só para a métrica wMAE (a loss é RMSE)
BINS = [-0.1, 3, 6, 12, 30, np.inf]                                   # classes de altura (m), fechadas à direita
BIN_NAMES = ["h_0_3", "h_3_6", "h_6_12", "h_12_30", "h_30p"]

RUN_TAG = "v8"
OUT_DIR = str(cs.OUT / f"runs_{RUN_TAG}")                                           # banco, artefatos e modelos desta versão
MLFLOW_URI = f"sqlite:///{OUT_DIR}/mlflow.db"                         # banco próprio: runs antigos ficam intactos
MLFLOW_EXPERIMENT = f"building_height_cnn_{RUN_TAG}"
MLFLOW_PORT = 5001                                                    # evita conflito com um `mlflow ui` antigo na 5000
RESUME_RUN_ID = None                                                  # força a retomada de um run pai específico
AUTO_RESUME = True                                                    # reabre o último run incompleto com a mesma config

TRAIN_SET = "augmented"                                               # "normal" | "augmented" | "balanced"
AUG_ROTS = (1, 2, 3)                                                  # rotações de 90° (1=90°, 2=180°, 3=270°)
OVERSAMPLE = {"h_12_30": 2, "h_30p": 6}                               # cópias extras por amostra (modo "balanced")

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
CONFIG = {"train_dir": TRAIN_DIR, "test_dir": TEST_DIR, "train_pattern": TRAIN_PATTERN, "fold_method": "folds_csv_block",
          "patch_px": PATCH_PX, "k_folds": K_FOLDS, "outer_fold": OUTER_FOLD, "inner_val_fold": INNER_VAL_FOLD,
          "seed": SEED, "train_set": TRAIN_SET, "aug_rots": list(AUG_ROTS), "oversample": OVERSAMPLE, "loss": "rmse_log1p"}
CONFIG_HASH = hashlib.md5(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:12]

mlflow.set_tracking_uri(MLFLOW_URI)
if mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT) is None:          # artefatos dentro de OUT_DIR
    mlflow.create_experiment(MLFLOW_EXPERIMENT, artifact_location=Path(OUT_DIR, "mlartifacts").resolve().as_uri())
mlflow.set_experiment(MLFLOW_EXPERIMENT)
if mlflow.active_run():
    mlflow.end_run(status="KILLED")                                   # run aberto por execução anterior: fica retomável

if RESUME_RUN_ID is None and AUTO_RESUME:                             # último run incompleto com a mesma configuração
    cand = mlflow.search_runs(experiment_names=[MLFLOW_EXPERIMENT], max_results=50, order_by=["attributes.start_time DESC"],
                              filter_string=f"tags.mlflow.runName = 'pipeline' and tags.config_hash = '{CONFIG_HASH}'")
    cand = cand[cand["status"] != "FINISHED"] if len(cand) else cand
    if len(cand):
        RESUME_RUN_ID = cand["run_id"].iloc[0]

if RESUME_RUN_ID:
    mlflow.start_run(run_id=RESUME_RUN_ID)
    client = mlflow.tracking.MlflowClient()
    stale = client.search_runs([mlflow.active_run().info.experiment_id],
                               filter_string=f"tags.mlflow.parentRunId = '{RESUME_RUN_ID}' and attributes.status = 'RUNNING'")
    for r in stale:
        client.set_terminated(r.info.run_id, "KILLED")                # iteração que ficou aberta quando o kernel caiu
    print("Retomando o run", RESUME_RUN_ID, f"({len(stale)} iteração(ões) interrompida(s) serão refeitas)")
else:
    mlflow.start_run(run_name="pipeline")
    print("Novo run", mlflow.active_run().info.run_id)

RUN_DIR = f"{OUT_DIR}/{mlflow.active_run().info.run_id}"              # modelos e tabelas só desta execução
os.makedirs(f"{RUN_DIR}/models", exist_ok=True)
os.makedirs(f"{RUN_DIR}/models_arch", exist_ok=True)

def out(name):
    return f"{RUN_DIR}/{name}"

mlflow.set_tags({"notebook_version": RUN_TAG, "config_hash": CONFIG_HASH, "train_dir": TRAIN_DIR, "test_dir": TEST_DIR,
                 "tf_version": tf.__version__, "n_gpus": str(len(gpus))})
mlflow.log_params({"seed": SEED, "patch_px": PATCH_PX, "k_folds": K_FOLDS, "outer_fold": OUTER_FOLD,
                   "inner_val_fold": INNER_VAL_FOLD, "epochs_max": EPOCHS, "w_ref": W_REF, "w_max": W_MAX,
                   "loss": "rmse_log1p", "train_set": TRAIN_SET, "aug_rots": str(AUG_ROTS), "oversample": str(OVERSAMPLE)})

## 3. Leitura dos dados de treino
Lê todos os `.tfrecord.gz` da pasta e mantém só `split == train`. O teste só é lido na última seção.

In [ ]:
spec = {"x": tf.io.FixedLenFeature([C_IN * S * S], tf.float32),
        "height_m": tf.io.FixedLenFeature([1], tf.float32),
        "block_x": tf.io.FixedLenFeature([1], tf.float32),
        "block_y": tf.io.FixedLenFeature([1], tf.float32),
        "sample_id": tf.io.FixedLenFeature([1], tf.string),
        "city": tf.io.FixedLenFeature([1], tf.string),
        "split": tf.io.FixedLenFeature([1], tf.string)}

def parse(rec):
    e = tf.io.parse_single_example(rec, spec)
    x = tf.transpose(tf.reshape(e["x"], [C_IN, S, S]), [1, 2, 0])      # (33, 33, 48)
    h = PATCH_PX // 2
    x = x[16 - h:17 + h, 16 - h:17 + h]                                # recorte central
    return (x, e["height_m"][0], e["block_x"][0], e["block_y"][0],
            e["sample_id"][0], e["city"][0], e["split"][0])

def load_numpy(folder, pattern):
    files = sorted(glob.glob(f"{folder}/{pattern}", recursive=True))
    assert files, f"nenhum arquivo em {folder}/{pattern}"
    print(f"{len(files)} arquivos em {folder}")
    ds = (tf.data.TFRecordDataset(files, compression_type="GZIP", num_parallel_reads=tf.data.AUTOTUNE)
            .map(parse, num_parallel_calls=tf.data.AUTOTUNE).batch(512).prefetch(1))
    cols = [np.concatenate(c) for c in zip(*ds.as_numpy_iterator())]
    d = dict(zip(["x", "h", "bx", "by", "sid", "city", "split"], cols))
    for k in ("sid", "city", "split"):
        d[k] = np.array([s.decode() for s in d[k]])
    _, keep = np.unique(d["sid"], return_index=True)                  # sample_id repetido entre arquivos
    if len(keep) < len(d["sid"]):
        print(f"{len(d['sid']) - len(keep)} registros duplicados removidos")
        d = {k: v[np.sort(keep)] for k, v in d.items()}
    return d

def subset(d, m):
    return {k: v[m] for k, v in d.items()}

train_all = load_numpy(TRAIN_DIR, TRAIN_PATTERN)
n_records = len(train_all["h"])
ds_train = subset(train_all, train_all["split"] == "train")           # descarta `unused` e `excluded`
train_all = None                                                      # libera a cópia completa
gc.collect()

print("registros nos arquivos:", n_records, "| train:", len(ds_train["h"]))
print("por cidade:", pd.Series(ds_train["city"]).value_counts().to_dict())
mlflow.log_param("n_records_train_files", n_records)

## 4. Split pelos folds dos modelos clássicos
Treino = folds restantes. Validação interna (`INNER_VAL_FOLD`) = early stopping, tuning e seleção. Hold-out (`OUTER_FOLD`) = só para relatar e gerar as predições out-of-fold.

In [ ]:
fold_of = cs.fold_of            # folds from data/folds.csv, shared with the classical models

fold = fold_of(ds_train["city"], ds_train["bx"], ds_train["by"], K_FOLDS)
assert set(np.unique(fold)) == set(range(K_FOLDS)), f"folds encontrados: {sorted(set(fold))}"
assert INNER_VAL_FOLD != OUTER_FOLD

tr_split = subset(ds_train, ~np.isin(fold, [OUTER_FOLD, INNER_VAL_FOLD]))
va = subset(ds_train, fold == INNER_VAL_FOLD)                         # validação interna
ho = subset(ds_train, fold == OUTER_FOLD)                             # hold-out externo
ds_train = None
gc.collect()
print("treino:", len(tr_split["h"]), "| validação interna:", len(va["h"]), "| hold-out:", len(ho["h"]))

## 5. Data augmentation e balanceamento (só no treino)
Cada variante é um plano de índices; os patches sintéticos são gerados direto no array final, na seção 6. `TRAIN_SET` escolhe o plano. Validação e hold-out nunca são alterados.

### 5.1 Treino normal

In [ ]:
tr_normal = tr_split                                                  # patches originais
plan_normal = [(np.arange(len(tr_normal["h"])), 0, False)]            # blocos: (índices, rotações de 90°, espelhar)
print("normal:", sum(len(b[0]) for b in plan_normal))

### 5.2 Treino com rotações sintéticas

In [ ]:
plan_aug = [(np.arange(len(tr_normal["h"])), k, False) for k in (0, *AUG_ROTS)]
print("augmented:", sum(len(b[0]) for b in plan_aug))

### 5.3 Prédios altos: balanceamento por oversampling (a loss continua RMSE)
Espaço para explorar outras estratégias contra o desbalanceamento.

In [ ]:
def class_of(h):
    return np.digitize(h, BINS[1:-1], right=True)

def plan_oversample(h, base_plan, plan):
    cls = class_of(h)
    extra = []
    for name, n in plan.items():
        idx = np.flatnonzero(cls == BIN_NAMES.index(name))
        extra += [(idx, j % 4, True) for j in range(n)]               # espelha + gira; acima de 4 cópias há repetição
    return base_plan + extra

plan_bal = plan_oversample(tr_normal["h"], plan_aug, OVERSAMPLE)
PLAN = {"normal": plan_normal, "augmented": plan_aug, "balanced": plan_bal}[TRAIN_SET]

h_plan = np.concatenate([tr_normal["h"][idx] for idx, _, _ in PLAN])
counts = np.bincount(class_of(h_plan), minlength=len(BIN_NAMES)).tolist()
print(TRAIN_SET, "->", len(h_plan), "| patches por classe:", dict(zip(BIN_NAMES, counts)))

## 6. Pré-processamento
Gera o array final em blocos (sem cópias intermediárias) e libera os arrays brutos. Para refazer esta seção, rode de novo a partir da seção 3.

In [ ]:
def channel_stats(X, chunk=2048):
    n = np.zeros(X.shape[-1]); s = np.zeros(X.shape[-1]); s2 = np.zeros(X.shape[-1])
    for i in range(0, len(X), chunk):
        xb = X[i:i + chunk].astype("float64")
        v = xb != NODATA
        xb = np.where(v, xb, 0.0)
        n += v.sum((0, 1, 2)); s += xb.sum((0, 1, 2)); s2 += (xb ** 2).sum((0, 1, 2))
    mean = s / n.clip(1)
    var = s2 / n.clip(1) - mean ** 2
    return mean.astype("float32"), np.sqrt(var.clip(0)).clip(1e-6).astype("float32")

MEAN, STD = channel_stats(tr_normal["x"])                             # só patches originais do treino

VALID_CH = [s * 12 + b for s in range(4) for b in (0, 2)]            # VV e B2 de cada estação

def prep(X, blocks=None, chunk=2048):
    if blocks is None:
        blocks = [(np.arange(len(X)), 0, False)]
    n, c = sum(len(b[0]) for b in blocks), X.shape[3]
    arr = np.empty((n, X.shape[1], X.shape[2], c + len(VALID_CH)), dtype="float32")
    pos = 0
    for idx, rot, flip in blocks:
        for s in range(0, len(idx), chunk):
            xb = X[idx[s:s + chunk]]
            if flip:
                xb = xb[:, :, ::-1]
            if rot:
                xb = np.rot90(xb, rot, axes=(1, 2))
            valid = xb != NODATA
            e = pos + len(xb)
            arr[pos:e, ..., :c] = np.where(valid, (xb - MEAN) / STD, 0.0)
            arr[pos:e, ..., c:] = valid[..., VALID_CH]
            pos = e
    return arr

X_tr, y_tr = prep(tr_normal["x"], PLAN), np.log1p(h_plan).astype("float32")      # alvo em log1p(m)
X_va, y_va = prep(va["x"]), np.log1p(va["h"]).astype("float32")
X_ho = prep(ho["x"])

for d in (tr_normal, va, ho):                                         # libera os arrays brutos
    d.pop("x", None)
tr_split = tr_normal = None
gc.collect()

mlflow.log_params({"n_train": len(y_tr), "n_val": len(y_va), "n_holdout": len(ho["h"]), "n_channels": X_tr.shape[-1]})
print("input shape:", X_tr.shape[1:], f"| X_tr = {X_tr.nbytes / 2 ** 30:.2f} GiB")

## 7. Pipeline, loss (RMSE) e funções de modelo

In [ ]:
def make_ds(X, y, training, batch=BATCH):                             # lê direto do array numpy: sem cópia nem buffer de shuffle
    n, rng = len(X), np.random.default_rng(SEED)
    def gen():
        idx = rng.permutation(n) if training else np.arange(n)       # nova ordem a cada época
        for s in range(0, n, batch):
            b = idx[s:s + batch]
            yield X[b], y[b]
    sig = (tf.TensorSpec((None, *X.shape[1:]), tf.float32), tf.TensorSpec((None,), tf.float32))
    return tf.data.Dataset.from_generator(gen, output_signature=sig).prefetch(tf.data.AUTOTUNE)

def rmse_loss(y_true, y_pred):                                        # RMSE em log1p(m), loss de todos os modelos
    y_true = tf.reshape(tf.cast(y_true, y_pred.dtype), (-1, 1))
    return tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)) + 1e-8)

def predict_m(m, X):
    p = np.concatenate([m.predict_on_batch(X[s:s + PRED_BATCH]) for s in range(0, len(X), PRED_BATCH)])
    return np.clip(np.expm1(np.asarray(p).reshape(-1)), 0, None)

def metrics(y, p):
    err = p - y
    w = np.minimum(1.0 + y / W_REF, W_MAX)                            # W_REF fixo: comparável entre trials
    r2 = 1 - (err ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return {"n": len(y), "MAE": np.abs(err).mean(), "wMAE": (w * np.abs(err)).sum() / w.sum(), "RMSE": np.sqrt((err ** 2).mean()), "bias": err.mean(), "R2": r2}

def class_rmse(y, p):
    idx = np.digitize(y, BINS[1:-1], right=True)
    return {f"RMSE_{BIN_NAMES[k]}": float(np.sqrt(((p[idx == k] - y[idx == k]) ** 2).mean()))
            for k in range(len(BIN_NAMES)) if (idx == k).any()}

def free_gpu():
    keras.backend.clear_session()
    gc.collect()

def reset_gpu_peak():
    try:
        tf.config.experimental.reset_memory_stats("GPU:0")
    except Exception:
        pass

def gpu_peak_mb():                                                    # pico de memória da GPU desde o último reset
    try:
        return {"gpu_peak_mb": tf.config.experimental.get_memory_info("GPU:0")["peak"] / 2 ** 20}
    except Exception:
        return {}

def gpu_in_use():                                                     # memória ocupada por tensores vivos na GPU
    try:
        return f"{tf.config.experimental.get_memory_info('GPU:0')['current'] / 2 ** 20:.0f} MB"
    except Exception:
        return "n/d"

def host_rss_mb():                                                    # RAM do processo do kernel
    try:
        import psutil
        return psutil.Process().memory_info().rss / 2 ** 20
    except Exception:
        return float("nan")

def log_resources():
    mlflow.log_metrics(gpu_peak_mb())
    if not np.isnan(host_rss_mb()):
        mlflow.log_metric("host_rss_mb", host_rss_mb())

def mlflow_epoch_cb():
    return keras.callbacks.LambdaCallback(
        on_epoch_end=lambda ep, logs: mlflow.log_metrics({k: float(v) for k, v in logs.items()}, step=ep))

# --- retomada: o MLflow é a fonte da verdade sobre o que já terminou
def finished_children(stage):
    run = mlflow.active_run().info
    return mlflow.search_runs(
        experiment_ids=[run.experiment_id], order_by=["attributes.start_time DESC"],
        filter_string=f"tags.mlflow.parentRunId = '{run.run_id}' and tags.stage = '{stage}' and attributes.status = 'FINISHED'")

def collect_rows(stage, make_row):
    rows = [make_row(q) for _, q in finished_children(stage).iterrows()]
    return pd.DataFrame([r for r in rows if r is not None])

def class_cols(q):
    return {c.replace("metrics.", ""): q[c] for c in q.index if c.startswith("metrics.val_RMSE_h_")}

def tuning_row(q):                                                    # baseline (trial 0) e trials do tuning
    name = q["tags.mlflow.runName"]
    t = 0 if name == "baseline" else int(name.split("_")[1])
    path = out(f"models/trial_{t}.keras")
    if not os.path.exists(path):
        return None                                                   # sem o modelo no disco: será refeito
    return {"trial": t, **{k: float(q[f"params.{k}"]) for k in ("lr", "base", "dense", "dropout", "batch")},
            "val_RMSE_m": q["metrics.val_RMSE"], "val_wMAE_m": q["metrics.val_wMAE"], "val_MAE_m": q["metrics.val_MAE"],
            **class_cols(q), "epochs": int(q["metrics.epochs_run"]), "model_path": path}

def arch_row(q):
    idx = int(q["tags.mlflow.runName"].split("_")[1])                 # run_name = arch_XX_<nome>
    path = out(f"models_arch/arch_{idx:02d}.keras")
    if not os.path.exists(path):
        return None
    return {"arch": idx, "name": q["params.arch_name"], "depth": int(q["params.depth"]),
            "convs": int(q["params.convs_per_level"]), "filters": q["params.filters"],
            "head": int(q["params.head_units"]), "dropout": float(q["params.dropout"]),
            "n_params": int(q["params.n_params"]), "val_RMSE_m": q["metrics.val_RMSE"],
            "val_MAE_m": q["metrics.val_MAE"], "val_wMAE_m": q["metrics.val_wMAE"], "val_R2": q["metrics.val_R2"],
            **class_cols(q), "epochs": int(q["metrics.epochs_run"]), "model_path": path}

def same_hp(row, hp):
    return all(abs(float(row[k]) - float(hp[k])) < 1e-12 for k in hp)

BASE_HP = {"lr": LR, "base": 32, "dense": 64, "dropout": 0.3, "batch": BATCH}

def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)

def build_model(input_shape, base=32, dense=64, dropout=0.3):
    i = layers.Input(input_shape)
    x = conv_block(i, base); x = conv_block(x, base); x = layers.MaxPool2D()(x)
    x = conv_block(x, 2 * base); x = conv_block(x, 2 * base); x = layers.MaxPool2D()(x)
    x = conv_block(x, 4 * base)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    return keras.Model(i, layers.Dense(1)(x))

## 8. Treino (modelo base)
Se o baseline já terminou neste run (retomada), não é treinado de novo.

In [ ]:
if len(collect_rows("baseline", tuning_row)):
    print("baseline já concluído neste run: nada a treinar")
else:
    model = ds_tr = ds_va = history = None
    try:
        free_gpu()
        reset_gpu_peak()
        model = build_model(X_tr.shape[1:])
        model.compile(optimizer=keras.optimizers.Adam(LR), loss=rmse_loss, metrics=["mae"])
        model.summary()
        ds_tr, ds_va = make_ds(X_tr, y_tr, True), make_ds(X_va, y_va, False)
        callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
                     keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-5),
                     mlflow_epoch_cb()]
        with mlflow.start_run(run_name="baseline", nested=True):
            mlflow.set_tag("stage", "baseline")
            mlflow.log_params(BASE_HP)
            history = model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS, callbacks=callbacks)
            p0 = predict_m(model, X_va)
            base_m, base_c = metrics(va["h"], p0), class_rmse(va["h"], p0)
            mlflow.log_metrics({f"val_{k}": float(v) for k, v in {**base_m, **base_c}.items()})
            mlflow.log_metric("epochs_run", len(history.history["loss"]))
            log_resources()
            model.save(out("models/trial_0.keras"))                   # o run só fica FINISHED com o modelo salvo
        print({k: round(float(v), 3) for k, v in {**base_m, **base_c}.items()})
    except tf.errors.ResourceExhaustedError:
        print("baseline: sem memória na GPU")
    finally:
        del model, ds_tr, ds_va, history
        free_gpu()
        print(f"   RAM do processo: {host_rss_mb():.0f} MB | GPU em uso: {gpu_in_use()}")

## 9. Tuning e seleção do modelo (random search, só treino e validação interna)
Loss: RMSE. Critério de seleção: RMSE em metros na validação interna. Retomável: trials já concluídos são pulados.

In [ ]:
N_TRIALS, TUNE_EPOCHS = 10, 40
SPACE = {"lr": [3e-4, 1e-3, 3e-3], "base": [16, 32, 48], "dense": [32, 64, 128],
         "dropout": [0.2, 0.3, 0.5], "batch": [64, 128, 256]}

In [ ]:
rng = np.random.default_rng(SEED)
prev = collect_rows("tuning", tuning_row)
prev_by_t = prev.drop_duplicates("trial").set_index("trial") if len(prev) else pd.DataFrame()
print(f"{len(prev_by_t)} de {N_TRIALS} trials já concluídos; restam {N_TRIALS - len(prev_by_t)}")

for t in range(1, N_TRIALS + 1):
    hp = {k: v[rng.integers(len(v))] for k, v in SPACE.items()}       # sorteio antes do skip: a sequência não muda
    if t in prev_by_t.index and same_hp(prev_by_t.loc[t], hp):
        continue
    m = h = cb = ds_t = ds_v = None
    try:
        free_gpu()
        reset_gpu_peak()
        m = build_model(X_tr.shape[1:], hp["base"], hp["dense"], hp["dropout"])
        m.compile(optimizer=keras.optimizers.Adam(hp["lr"]), loss=rmse_loss)
        cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
              keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5),
              mlflow_epoch_cb()]
        ds_t = make_ds(X_tr, y_tr, True, int(hp["batch"]))
        ds_v = make_ds(X_va, y_va, False, int(hp["batch"]))

        with mlflow.start_run(run_name=f"trial_{t}", nested=True):
            mlflow.set_tag("stage", "tuning")
            mlflow.log_params({k: float(v) for k, v in hp.items()})
            h = m.fit(ds_t, validation_data=ds_v, epochs=TUNE_EPOCHS, callbacks=cb, verbose=0)
            pv = predict_m(m, X_va)
            r, rc = metrics(va["h"], pv), class_rmse(va["h"], pv)
            mlflow.log_metrics({f"val_{k}": float(v) for k, v in {**r, **rc}.items()})
            mlflow.log_metric("epochs_run", len(h.history["loss"]))
            log_resources()
            m.save(out(f"models/trial_{t}.keras"))                    # o run só fica FINISHED com o modelo salvo
        print(f"trial {t}/{N_TRIALS}  val RMSE = {r['RMSE']:.3f} m  (MAE = {r['MAE']:.3f} m)  {hp}")
        print("   RMSE por classe:", {k: round(v, 2) for k, v in rc.items()})
    except tf.errors.ResourceExhaustedError:                          # o run fica FAILED no MLflow, com os parâmetros
        print(f"trial {t}/{N_TRIALS}  sem memória na GPU, pulado  {hp}")
    finally:
        del m, h, cb, ds_t, ds_v                                      # libera o trial mesmo em caso de erro
        free_gpu()
        print(f"   RAM do processo: {host_rss_mb():.0f} MB | GPU em uso: {gpu_in_use()}")

tuning = (pd.concat([collect_rows("baseline", tuning_row), collect_rows("tuning", tuning_row)])
          .drop_duplicates("trial").sort_values("val_RMSE_m").reset_index(drop=True))
tuning.round(4)

In [ ]:
free_gpu()
shutil.copy(tuning.model_path[0], out("cnn_best.keras"))                    # melhor candidato pelo RMSE
model = keras.models.load_model(out("cnn_best.keras"), compile=False)      # modelo selecionado
print(f"Selecionado: trial {int(tuning.trial[0])} | val RMSE = {tuning.val_RMSE_m[0]:.3f} m")

best = tuning.iloc[0].drop([c for c in tuning.columns if c.startswith("val_") or c == "epochs"]).to_dict()
mlflow.set_tags({f"best_{k}": str(v) for k, v in best.items()})
tuning.to_csv(out("tuning_results.csv"), index=False)
mlflow.log_artifact(out("tuning_results.csv"))

## 10. Busca de arquiteturas em U (20 arquiteturas, loss RMSE)
Independente do tuning da seção 9: aqui varia a estrutura da rede (profundidade, convoluções por nível, filtros por nível).
A loss é o RMSE em log1p(m); a seleção usa o RMSE em metros na validação interna. Retomável: arquiteturas já concluídas são puladas.

In [ ]:
import itertools

N_ARCH, ARCH_EPOCHS, ARCH_LR, ARCH_BATCH = 20, 40, 1e-3, 128

def build_unet(input_shape, depth, convs, filters, head, dropout):
    h, w, _ = input_shape
    c = h // 2                                                        # pixel central do patch
    p = (-h) % (2 ** depth)                                           # padding para dividir por 2**depth
    i = layers.Input(input_shape)
    x = layers.ZeroPadding2D(((0, p), (0, p)))(i) if p else i
    skips = []
    for f in filters[:-1]:                                            # encoder
        for _ in range(convs):
            x = conv_block(x, f)
        skips.append(x)
        x = layers.MaxPool2D()(x)
    for _ in range(convs):                                            # gargalo
        x = conv_block(x, filters[-1])
    for f, s in zip(filters[-2::-1], skips[::-1]):                    # decoder com skip connections
        x = layers.UpSampling2D()(x)
        x = layers.Concatenate()([x, s])
        for _ in range(convs):
            x = conv_block(x, f)
    if p:
        x = layers.Cropping2D(((0, p), (0, p)))(x)
    center = layers.Flatten()(layers.Cropping2D(((c, h - c - 1), (c, w - c - 1)))(x))
    x = layers.Concatenate()([center, layers.GlobalAveragePooling2D()(x)])   # pixel central + contexto
    x = layers.Dense(head, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    return keras.Model(i, layers.Dense(1)(x))

# espaço: profundidade, convs por nível, filtros base, multiplicador por nível, neurônios da cabeça, dropout
ARCH_SPACE = list(itertools.product([1, 2, 3], [1, 2, 3], [8, 16, 32, 48], [1, 2], [32, 64, 128], [0.0, 0.2, 0.4]))
pick = np.random.default_rng(SEED).choice(len(ARCH_SPACE), N_ARCH, replace=False)

ARCHS = []
for idx in pick:
    d, c, b, mu, hd, dr = ARCH_SPACE[idx]
    f = [min(b * mu ** l, 256) for l in range(d + 1)]                 # filtros por nível (último = gargalo)
    ARCHS.append({"name": f"d{d}_c{c}_f{b}x{mu}_h{hd}", "depth": d, "convs": c, "base": b, "mult": mu,
                  "filters": f, "head": hd, "dropout": dr})
pd.DataFrame(ARCHS)

In [ ]:
prev = collect_rows("architecture", arch_row)
done = set(zip(prev["name"], prev["dropout"])) if len(prev) else set()   # arquiteturas já concluídas neste run pai
print(f"{len(done)} de {N_ARCH} já concluídas; restam {N_ARCH - len(done)}")

for i, a in enumerate(ARCHS, 1):
    if (a["name"], float(a["dropout"])) in done:
        continue
    m = h = cb = ds_t = ds_v = None
    try:
        free_gpu()
        reset_gpu_peak()
        m = build_unet(X_tr.shape[1:], a["depth"], a["convs"], a["filters"], a["head"], a["dropout"])
        m.compile(optimizer=keras.optimizers.Adam(ARCH_LR), loss=rmse_loss)
        n_params = m.count_params()
        cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
              keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5),
              mlflow_epoch_cb()]
        ds_t = make_ds(X_tr, y_tr, True, ARCH_BATCH)
        ds_v = make_ds(X_va, y_va, False, ARCH_BATCH)

        with mlflow.start_run(run_name=f"arch_{i:02d}_{a['name']}", nested=True):
            mlflow.set_tag("stage", "architecture")
            mlflow.log_params({"arch_name": a["name"], "depth": a["depth"], "convs_per_level": a["convs"],
                               "filters": str(a["filters"]), "head_units": a["head"], "dropout": a["dropout"],
                               "n_params": n_params, "lr": ARCH_LR, "batch": ARCH_BATCH, "loss": "rmse_log1p"})
            h = m.fit(ds_t, validation_data=ds_v, epochs=ARCH_EPOCHS, callbacks=cb, verbose=0)
            pv = predict_m(m, X_va)
            r, rc = metrics(va["h"], pv), class_rmse(va["h"], pv)
            mlflow.log_metrics({f"val_{k}": float(v) for k, v in {**r, **rc}.items()})
            mlflow.log_metric("epochs_run", len(h.history["loss"]))
            log_resources()
            m.save(out(f"models_arch/arch_{i:02d}.keras"))            # o run só fica FINISHED com o modelo salvo
        print(f"arch {i}/{N_ARCH}  {a['name']}  params = {n_params:,}  val RMSE = {r['RMSE']:.3f} m  (MAE = {r['MAE']:.3f} m)")
        print("   RMSE por classe:", {k: round(v, 2) for k, v in rc.items()})
    except tf.errors.ResourceExhaustedError:                          # o run fica FAILED e é refeito na retomada
        print(f"arch {i}/{N_ARCH}  {a['name']}  sem memória na GPU, pulada")
    finally:
        del m, h, cb, ds_t, ds_v
        free_gpu()
        print(f"   RAM do processo: {host_rss_mb():.0f} MB | GPU em uso: {gpu_in_use()}")

In [ ]:
arch_tbl = collect_rows("architecture", arch_row).drop_duplicates("arch").sort_values("val_RMSE_m").reset_index(drop=True)
arch_tbl.to_csv(out("architecture_results.csv"), index=False)
shutil.copy(arch_tbl.model_path[0], out("unet_best.keras"))                # melhor arquitetura pelo RMSE na validação interna

b = arch_tbl.iloc[0]
mlflow.set_tags({"best_unet_name": b["name"], "best_unet_filters": b["filters"], "best_unet_depth": str(int(b["depth"])),
                 "best_unet_convs": str(int(b["convs"])), "best_unet_head": str(int(b["head"])),
                 "best_unet_n_params": str(int(b["n_params"]))})
mlflow.log_metrics({"best_unet_val_RMSE": float(b["val_RMSE_m"]), "best_unet_val_MAE": float(b["val_MAE_m"]),
                    "best_unet_val_wMAE": float(b["val_wMAE_m"])})
mlflow.log_metrics({f"best_unet_{c}": float(b[c]) for c in arch_tbl.columns if c.startswith("val_RMSE_h_")})
mlflow.log_artifact(out("architecture_results.csv"))
mlflow.log_artifact(out("unet_best.keras"))

USE_UNET_AS_FINAL = False                                             # True: a U-Net vira `model` nas seções seguintes
print(f"RMSE de validação interna: tuning {tuning.val_RMSE_m[0]:.3f} m | U-Net {arch_tbl.val_RMSE_m[0]:.3f} m")
if USE_UNET_AS_FINAL:
    free_gpu()
    model = keras.models.load_model(out("unet_best.keras"), compile=False)
mlflow.set_tag("final_model", "unet" if USE_UNET_AS_FINAL else "cnn_tuning")
arch_tbl.round(4)

## 11. Avaliação: validação interna e hold-out (modelo selecionado)
A validação interna já foi usada na seleção; o hold-out é o número a relatar.

In [ ]:
p_va = predict_m(model, X_va)
p_ho = predict_m(model, X_ho)
sel_m, sel_c = metrics(va["h"], p_va), class_rmse(va["h"], p_va)
ho_m, ho_c = metrics(ho["h"], p_ho), class_rmse(ho["h"], p_ho)
mlflow.log_metrics({f"sel_val_{k}": float(v) for k, v in {**sel_m, **sel_c}.items()})
mlflow.log_metrics({f"ho_{k}": float(v) for k, v in {**ho_m, **ho_c}.items()})
pd.DataFrame({f"val interna (fold {INNER_VAL_FOLD}, seleção)": {**sel_m, **sel_c},
              f"hold-out (fold {OUTER_FOLD})": {**ho_m, **ho_c}}).T.round(3)

In [ ]:
df = pd.DataFrame({"h": ho["h"], "p": p_ho})
df["faixa"] = pd.cut(df.h, BINS)
bin_table = df.groupby("faixa", observed=True).apply(
    lambda g: pd.Series({"n": len(g), "MAE": (g.p - g.h).abs().mean(), "bias": (g.p - g.h).mean()})).round(3)
bin_table.to_csv(out("ho_by_height_bin.csv"))
mlflow.log_artifact(out("ho_by_height_bin.csv"))

oof = pd.DataFrame({"sample_id": ho["sid"], "fold": OUTER_FOLD, "cnn_pred": p_ho})   # predições out-of-fold (m)
oof.to_csv(out(f"oof_fold{OUTER_FOLD}.csv"), index=False)
mlflow.log_artifact(out(f"oof_fold{OUTER_FOLD}.csv"))
bin_table

## 12. Salvar artefatos

In [ ]:
np.savez(out("norm_stats.npz"), mean=MEAN, std=STD, patch_px=PATCH_PX)
json.dump(CONFIG, open(out("config.json"), "w"), indent=1)
mlflow.log_artifact(out("norm_stats.npz"))
mlflow.log_artifact(out("config.json"))
mlflow.log_artifact(out("cnn_best.keras"))                                   # modelo selecionado

## 13. Dashboard do MLflow (opcional)

In [ ]:
import subprocess, sys
from IPython.display import IFrame

ui = subprocess.Popen([sys.executable, "-m", "mlflow", "ui", "--backend-store-uri", MLFLOW_URI, "--port", str(MLFLOW_PORT)])
IFrame(f"http://127.0.0.1:{MLFLOW_PORT}", width="100%", height=700)

## 14. Teste final
Executar uma única vez, depois de fechar o modelo. Nada aqui retroalimenta treino, tuning ou seleção.

In [ ]:
test = load_numpy(TEST_DIR, TEST_PATTERN)
X_te = prep(test["x"])                                                # usa MEAN/STD do treino
test.pop("x")
p_te = predict_m(model, X_te)

rows = {"test (todos)": metrics(test["h"], p_te)}

csv_files = glob.glob(DIST_GLOB)
if csv_files:
    dist = pd.concat([pd.read_csv(f) for f in csv_files])
    clean = np.isin(test["sid"], dist.sample_id[dist.cheb_to_train_m >= PATCH_PX * 10].values)
    rows["test (clean)"] = metrics(test["h"][clean], p_te[clean])
else:
    print(f"ATENÇÃO: nenhum CSV de distâncias em {DIST_GLOB}; teste 'clean' não calculado")

for name, r in rows.items():
    key = name.replace("test (", "test_").replace(")", "")
    mlflow.log_metrics({f"{key}_{k}": float(v) for k, v in r.items()})
mlflow.end_run()                                                      # run FINISHED: não será retomado de novo

pd.DataFrame(rows).T.round(3)